# 📊 PyTorch & TensorBoard ile Deney Takibi (Experiment Tracking) ve Modüler Eğitim

Bu notebook'ta, derin öğrenme projelerinde farklı hiperparametre ve model mimarilerini bilimsel olarak karşılaştırmak için kullanılan **TensorBoard (`SummaryWriter`)** entegrasyonunu ve **PyCharm modüler kod yapısını** (`data_setup.py`, `model_builder.py`, `engine.py`, `train.py`) inceliyoruz.

---

### 🗺️ İçindekiler
1. [Gerekli Kütüphaneler ve TensorBoard Kurulumu](#1-gerekli-kütüphaneler-ve-tensorboard-kurulumu)
2. [`SummaryWriter` (Kayıt Cihazı) Mekanizması](#2-summarywriter-kayıt-cihazı-mekanizması)
3. [Modüler Veri Hattı: `create_dataloaders()`](#3-modüler-veri-hattı-create_dataloaders)
4. [Modüler Model Mimarisi: `DesertClassifier` (TinyVGG)](#4-modüler-model-mimarisi-desertclassifier-tinyvgg)
5. [TensorBoard Entegreli Eğitim Motoru: `train_step()`, `test_step()` ve `train()`](#5-tensorboard-entegreli-eğitim-motoru-train_step-test_step-ve-train)
6. [Hiperparametre Deneylerinin Kurulumu (`hidden_units=10` vs `hidden_units=32`)](#6-hiperparametre-deneylerinin-kurulumu-hidden_units10-vs-hidden_units32)
7. [Deney 1: `hidden_units_10` Model Eğitimi](#7-deney-1-hidden_units_10-model-eğitimi)
8. [Deney 2: `hidden_units_32` Model Eğitimi](#8-deney-2-hidden_units_32-model-eğitimi)
9. [TensorBoard Canlı Kokpitinin Başlatılması & Sonuç Analizi](#9-tensorboard-canlı-kokpitinin-başlatılması--sonuç-analizi)


## 1. Gerekli Kütüphaneler ve TensorBoard Kurulumu


In [1]:
import torch
from torch import nn

In [2]:
!pip install tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 674.3 kB/s  0:00:087.8 kB/s eta 0:00:01:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 402.1 kB/s  0:00:38371.9 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [tensorboard] 3/4 [tensorboard]


## 2. `SummaryWriter` (Kayıt Cihazı) Mekanizması

`torch.utils.tensorboard.SummaryWriter`, eğitim sırasında üretilen kayıp (`loss`), doğruluk (`accuracy`) ve model grafiği (`graph`) verilerini diskteki `runs/` klasörüne kaydeden ana sınıftır.


In [3]:
from torch.utils.tensorboard import SummaryWriter

In [4]:
writer = SummaryWriter()

## 3. Modüler Veri Hattı: `create_dataloaders()`

PyCharm üzerinde `data_setup.py` dosyasında kurguladığımız modüler veri yükleme fonksiyonu:
- Klasör yollarını (`train_dir`, `test_dir`) ve dönüşüm nesnesini (`transform`) alır.
- `datasets.ImageFolder` ile görüntüleri etiketler.
- Eğitim ve test `DataLoader` nesneleri ile sınıf isimlerini (`class_names`) döndürür.


In [5]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()
# NUM_WORKERS = 0 # Apple Silicon (Mac) üzerinde multiprocessing hatası alınırsa 0 yapılabilir

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int = NUM_WORKERS
):
    # ImageFolder ile klasör yapısından veri setlerini oluştur
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    # Sınıf isimlerini al
    class_names = train_data.classes

    # Veri setlerini DataLoader paketlerine dönüştür
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True, # Eğitim verisini her epoch'ta karıştır
        num_workers=num_workers,
        pin_memory=True,
    )
    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False, # Test verisini karıştırmaya gerek yoktur
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names


## 4. Modüler Model Mimarisi: `DesertClassifier` (TinyVGG)

PyCharm üzerinde `model_builder.py` dosyasında tanımlanan **TinyVGG** mimarisi.  
$64 \times 64$ boyutundaki görseller iki kez `MaxPool2d(kernel_size=2)` katmanından geçtiği için uzamsal boyut $64 \to 32 \to 16$ olur ve `nn.Linear` katmanı `hidden_units * 16 * 16` girdi alır:


In [6]:
class DesertClassifier(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)  # Varsayılan stride değeri kernel_size (2) ile aynıdır
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 16 * 16,
                      out_features=output_shape)
        )

    def forward(self, x: torch.Tensor):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))


## 5. TensorBoard Entegreli Eğitim Motoru: `train_step()`, `test_step()` ve `train()`

PyCharm'daki `engine.py` modülüne **TensorBoard (`SummaryWriter`)** kayıt mekanizmasını ekliyoruz:
- **`writer = SummaryWriter(log_dir=f"runs/{experiment_name}")`:** Döngü başlamadan **önce bir kez** oluşturulur ve her deneyi kendi klasörüne (`runs/hidden_units_10` vb.) yazar.
- **`writer.add_scalars()`:** Her epoch sonunda eğitim ve test kayıp/doğruluk değerlerini grafiklere işler.
- **`writer.add_graph()`:** Eğitim tamamlandığında modelin mimari akış şemasını sadece **bir kez** kaydeder ve `writer.close()` ile kayıt kapatılır.


In [7]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):
    # Modeli eğitim moduna al
    model.train()

    # Eğitim kayıp ve doğruluk sayaçlarını başlat
    train_loss, train_acc = 0, 0

    # DataLoader mini-batch'leri üzerinde döngü
    for batch, (X, y) in enumerate(dataloader):

        # 1. İleri Yayılım (Forward pass)
        y_pred = model(X)

        # 2. Kaybı hesapla ve biriktir
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Gradyanları sıfırla
        optimizer.zero_grad()

        # 4. Geriye Yayılım (Backpropagation)
        loss.backward()

        # 5. Ağırlıkları güncelle (Optimizer step)
        optimizer.step()

        # Tüm batch'ler boyunca doğruluk (accuracy) metriğini hesapla ve topla
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    # Batch başına ortalama kayıp ve doğruluğu hesapla
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    # Modeli değerlendirme (eval) moduna al
    model.eval()

    # Test kayıp ve doğruluk sayaçlarını başlat
    test_loss, test_acc = 0, 0

    # Çıkarım (inference) modunu aç (gradyan takibini kapat)
    with torch.inference_mode():
        # Test DataLoader batch'leri üzerinde döngü
        for batch, (X, y) in enumerate(dataloader):
            # 1. İleri Yayılım (Forward pass)
            test_pred_logits = model(X)

            # 2. Kaybı hesapla ve biriktir
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Doğruluk (accuracy) oranını hesapla ve biriktir
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    # Batch başına ortalama test kaybı ve doğruluğunu hesapla
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5,
          experiment_name: str = "experiment"):
    # 1. Sonuç sözlüğünü oluştur
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
               }

    # 2. TensorBoard SummaryWriter nesnesini döngüden ÖNCE bir kez başlat
    writer = SummaryWriter(log_dir=f"runs/{experiment_name}")

    # 3. Belirtilen epoch sayısı boyunca eğitim ve test adımlarını çalıştır
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn)

        # 4. Epoch sonuçlarını ekrana yazdır
        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # 5. Sonuç sözlüğünü güncelle
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

        # 6. Kayıp (Loss) metriklerini TensorBoard'a kaydet
        writer.add_scalars(main_tag="Loss",
                           tag_scalar_dict={"train_loss": train_loss,
                                            "test_loss": test_loss},
                           global_step=epoch)

        # 7. Doğruluk (Accuracy) metriklerini TensorBoard'a kaydet
        writer.add_scalars(main_tag="Accuracy",
                           tag_scalar_dict={"train_acc": train_acc,
                                            "test_acc": test_acc},
                           global_step=epoch)

    # 8. Modelin mimari grafiğini döngü bittikten sonra BİR KEZ kaydet ve yazıcıyı kapat
    writer.add_graph(model=model,
                     input_to_model=torch.randn(32, 3, 64, 64))
    writer.close()

    # 9. Eğitim sonunda sonuçları döndür
    return results


## 6. Hiperparametre Deneylerinin Kurulumu (`hidden_units=10` vs `hidden_units=32`)

Aynı veri seti üzerinde filtre/nöron sayısının (`hidden_units`) öğrenme üzerindeki etkisini kıyaslamak için iki ayrı model tanımlıyoruz:
- **`model`:** `hidden_units = 10` (Her evrişim katmanında 10 filtre)
- **`model2`:** `hidden_units = 32` (Her evrişim katmanında 32 filtre)


In [9]:
NUM_EPOCHS = 2
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Veri klasör yolları
train_dir = "data/desert101/train"
test_dir = "data/desert101/test"

# Görsel dönüşümleri (64x64 boyutlandırma ve tensöre çevirme)
data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# create_dataloaders fonksiyonu ile DataLoader'ları oluştur
train_dataloader, test_dataloader, class_names = create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# Deney 1 Modeli: 10 gizli birim (filtre)
model = DesertClassifier(
    input_shape=3,
    hidden_units=10,
    output_shape=len(class_names)
)

# Deney 2 Modeli: 32 gizli birim (filtre)
model2 = DesertClassifier(
    input_shape=3,
    hidden_units=32,
    output_shape=len(class_names)
)


In [10]:
loss_fn = torch.nn.CrossEntropyLoss()


## 7. Deney 1: `hidden_units_10` Model Eğitimi

İlk modelimizi `experiment_name="hidden_units_10"` etiketiyle eğitiyoruz. Kayıtlar `runs/hidden_units_10` dizinine yazılır:


In [11]:
train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer=torch.optim.Adam(model.parameters(), lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    experiment_name="hidden_units_10"
)


/opt/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1 | train_loss: 1.3953 | train_acc: 0.2464 | test_loss: 1.3872 | test_acc: 0.1979


/opt/anaconda3/lib/python3.14/site-packages/torch/jit/_trace.py:1000: FutureWarning: `torch.jit.trace` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/torch/jit/_trace.py:1139: FutureWarning: `torch.jit.trace_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Epoch: 2 | train_loss: 1.3881 | train_acc: 0.2482 | test_loss: 1.3948 | test_acc: 0.2083


{'train_loss': [1.395286226272583, 1.3881129026412964],
 'train_acc': [0.24642857142857144, 0.24821428571428572],
 'test_loss': [1.3872300783793132, 1.394789179166158],
 'test_acc': [0.19791666666666666, 0.20833333333333334]}

## 8. Deney 2: `hidden_units_32` Model Eğitimi

İkinci modelimizi `experiment_name="hidden_units_32"` etiketiyle eğitiyoruz. Kayıtlar `runs/hidden_units_32` dizinine yazılır:


In [12]:
train(
    model=model2,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer=torch.optim.Adam(model2.parameters(), lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    experiment_name="hidden_units_32"
)


Epoch: 1 | train_loss: 1.3932 | train_acc: 0.2420 | test_loss: 1.3701 | test_acc: 0.4014
Epoch: 2 | train_loss: 1.3508 | train_acc: 0.3076 | test_loss: 1.3805 | test_acc: 0.2548


{'train_loss': [1.3932070136070251, 1.3507987022399903],
 'train_acc': [0.24196428571428572, 0.30758928571428573],
 'test_loss': [1.3701157172520955, 1.3804521163304646],
 'test_acc': [0.4014423076923077, 0.2548076923076923]}

## 9. TensorBoard Canlı Kokpitinin Başlatılması & Sonuç Analizi

`%tensorboard --logdir runs` sihirli komutu (magic command) ile `runs/` klasöründeki tüm deneyleri tek bir interaktif grafik ekranında açıyoruz:


In [13]:
%load_ext tensorboard
%tensorboard --logdir runs # "runs/" klasöründeki tüm deney kayıtlarını interaktif panelde başlatır


### 📊 Deney Sonuçlarının Birebir Karşılaştırması (2 Epoch Hızlı Test):

| Deney Adı (`experiment_name`) | Filtre Sayısı (`hidden_units`) | Epoch 1 Test Acc | Epoch 2 Train Loss | Epoch 2 Train Acc | Epoch 2 Test Loss | Epoch 2 Test Acc |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **`hidden_units_10` (`model`)** | `10` | `%19.79` (`0.1979`) | `1.3881` | `%24.82` (`0.2482`) | `1.3948` | `%20.83` (`0.2083`) |
| **`hidden_units_32` (`model2`)** | `32` | **`%40.14`** (`0.4014`) | **`1.3508`** | **`%30.76`** (`0.3076`) | **`1.3805`** | **`%25.48`** (`0.2548`) |

---

### 📌 Çıkarılan Mühendislik Dersleri:
1. **Deney Takibi (Experiment Tracking):** TensorBoard sayesinde farklı hiperparametrelere (`hidden_units=10` vs `32`) sahip modellerin kayıp ve doğruluk eğrilerini aynı grafik üzerinde üst üste bindirerek kıyaslayabiliriz.
2. **Kapasite Etkisi:** Filtre sayısını 10'dan 32'ye çıkarmak, daha ilk 2 epoch'ta eğitim doğruluğunu `%24.82`'den `%30.76`'ya, ilk epoch test doğruluğunu ise `%19.79`'dan `%40.14`'e taşıdı.
3. **Performans Optimizasyonu:** `SummaryWriter` nesnesini oluşturma ve `add_graph()` ile model mimarisini çizdirme işlemleri `for epoch` döngüsünün **dışında** tutulmalıdır; döngü içinde sadece `add_scalars()` çalıştırılmalıdır.
